In [0]:
%sql
CREATE CATALOG IF NOT EXISTS happiness;

CREATE SCHEMA IF NOT EXISTS happiness.landing;
CREATE SCHEMA IF NOT EXISTS happiness.bronze;
CREATE SCHEMA IF NOT EXISTS happiness.silver;
CREATE SCHEMA IF NOT EXISTS happiness.gold;
CREATE SCHEMA IF NOT EXISTS happiness.meta;

-- Volume: almacenamiento para los archivos crudos (xlsx, csv.gz, json) antes de parsearlos
CREATE VOLUME IF NOT EXISTS happiness.landing.archivos;

In [0]:
%sql
WITH fuente_datos AS (
  SELECT * FROM VALUES
    ('ILOSTAT', 'HOW_TEMP_SEX_NB_A',    'Horas semanales habituales por persona ocupada',      'Tiempo de trabajo',  'horas/semana', false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EMP_NIFL_SEX_RT_A',    'Tasa de empleo informal',                              'Informalidad',       '%',            false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EMP_XTRU_SEX_RT_A',    'Subempleo por insuficiencia de horas',                 'Subempleo',          '%',            false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EIP_NEET_SEX_RT_A',    'Jovenes que ni estudian ni trabajan (NEET)',           'Juventud',           '%',            true,  'excepcion D4/D6: 75.5%, cerca del umbral y unico indicador de juventud', DATE'2026-11-10'),
    ('ILOSTAT', 'LUU_2LU2_SEX_RT_A',    'Tasa de subutilizacion de la fuerza laboral (LU2)',    'Calidad del empleo', '%',            true,  'supera 80% de cobertura; compuesto que incluye desempleo y fuerza laboral potencial', DATE'2026-11-10'),
    ('ILOSTAT', 'HOW_TEMP_SEX_AGE_NB_A','Horas semanales por sexo y edad',                      'Tiempo de trabajo',  'horas/semana', false, 'cobertura < 80%; redundante con HOW_TEMP_SEX_NB_A', DATE'2026-11-10')
  AS t(fuente, codigo, nombre, tema, unidad, incluido, motivo, fecha_decision)
)
MERGE INTO happiness.meta.config_indicadores AS destino
USING fuente_datos
ON destino.fuente = fuente_datos.fuente AND destino.codigo = fuente_datos.codigo
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM happiness.meta.config_indicadores WHERE incluido = true;